# Candlestick patterns vs. machine learning — review notebook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/goopernooper/candlestick-vs-ml/blob/main/notebooks/colab_review.ipynb)

This notebook runs the project from its GitHub repository, [goopernooper/candlestick-vs-ml](https://github.com/goopernooper/candlestick-vs-ml). The code lives in
ordinary Python files under `src/`; the notebook installs the exact library versions the study used, runs the test
suite, recomputes the statistical tests and figures from the committed results, and shows the main tables. Every
file can be opened from the **Files** panel on the left (folder `candlestick-vs-ml`).

**Part A** needs nothing but this notebook and takes about 5 minutes. **Part B** re-runs the whole experiment from
the raw price data, which is not in the public repository (Yahoo Finance's terms do not allow redistributing it), so
it needs a copy of the data snapshot shared privately through Google Drive.

Use **Runtime → Run all** for Part A. Part B is switched off until its settings are filled in.

## Part A — review without the raw data

### A1. Set up

Clones the repository and installs the pinned versions from `requirements.txt` into a separate environment
(`/content/venv`), so Colab's own preinstalled packages are left alone and nothing needs a runtime restart. All
project code runs with that environment's Python, `PY`.

In [ ]:
import os
import sys

# Project scripts draw their figures to files; this keeps them from using the notebook's plotting backend.
os.environ["MPLBACKEND"] = "Agg"

REPO_URL = "https://github.com/goopernooper/candlestick-vs-ml.git"
ROOT = "/content/candlestick-vs-ml"
PY = "/content/venv/bin/python"

if os.path.isdir(ROOT):
    !git -C {ROOT} pull -q
else:
    !git clone -q {REPO_URL} {ROOT}
%cd {ROOT}

!pip install -q uv
if not os.path.exists(PY):
    !uv venv -q --python {sys.executable} /content/venv
!uv pip install -q --python {PY} -r requirements.txt

!git log -1 --format='Code version: commit %h, %ad' --date=short
!{PY} -c "import sys, numpy, pandas, sklearn, lightgbm, talib; print('Python', sys.version.split()[0], '| numpy', numpy.__version__, '| pandas', pandas.__version__, '| scikit-learn', sklearn.__version__, '| LightGBM', lightgbm.__version__, '| TA-Lib', talib.__version__)"

### A2. Run the test suite

About 340 automated tests check the encodings for look-ahead bias, the walk-forward splits and purging, the holdout
lock, the models (including the synthetic planted-pattern checks of Section 5.3), the metrics, the backtest and the
statistical tests. The few tests that read the raw data report as *skipped* here; with the data present, all of them
run.

In [ ]:
!{PY} -m pytest -q -p no:cacheprovider

### A3. Development results, 2005–2022

The committed results of the main run (`results/v4`): every model, sorted by MCC. Trading columns are the
long/short strategy at 5 basis points per side.

In [ ]:
import pandas as pd
from IPython.display import Image, Markdown, display

pd.options.display.float_format = "{:.4f}".format
summary = pd.read_csv("results/v4/summary.csv")
cols = {"model": "model", "mcc": "MCC", "balanced_accuracy": "balanced acc.", "accuracy": "accuracy",
        "predicted_up_share": "share predicted up", "ls_ann_return": "L/S annual return", "ls_sharpe": "L/S Sharpe"}
display(summary[list(cols)].rename(columns=cols).sort_values("MCC", ascending=False).reset_index(drop=True))

### A4. Recompute the statistical tests

Diebold–Mariano tests, the Model Confidence Set and the SPA test (Section 8), recomputed from the committed daily
loss series with 10,000 bootstrap replications. This takes under a minute; the output should match the paper (for
example, SPA *p* = 0.731 against reversal, and an economic Model Confidence Set of `always_long`, `majority`,
`svm_b` and `svm_ohlc`).

In [ ]:
!{PY} src/significance.py v4

### A5. The 2023–2025 holdout and the confirmatory test

The holdout was used once, after the analysis plan was frozen (git tag `analysis-plan-frozen`). H1 is the
pre-registered primary test; the H2 rows are the per-model tests with the Holm–Bonferroni correction.

In [ ]:
holdout = pd.read_csv("results/holdout/summary.csv")
display(holdout[["model", "mcc", "balanced_accuracy", "ls_sharpe"]].sort_values("mcc", ascending=False).reset_index(drop=True))
display(pd.read_csv("results/holdout/confirmatory_tests.csv"))

### A6. Figures

Regenerates the paper's figures from the committed results and shows them.

In [ ]:
!{PY} src/figures.py v4
for name in ["figure_grid", "figure_cost_curve", "figure_coverage", "figure_confident", "figure_episodes"]:
    path = f"results/v4/{name}.png"
    if os.path.exists(path):
        display(Markdown(f"**{name}.png**"))
        display(Image(path, width=900))

### A7. Replication of Caginalp and Laurent (1998)

Their eight three-day reversal patterns, coded from their definitions and tested on 2002–2022 (Section 9.9).

In [ ]:
rep = pd.read_csv("results/caginalp_replication.csv")
display(rep[(rep["definition"] == "paper") & (rep["outcome"] == "cl") & (rep["period"] == "2002-2022")]
        [["pattern", "direction", "n", "success", "base", "z_binomial", "t_dates", "net_bps"]].reset_index(drop=True))

### A8. Where things are in the code

| File | What it does | Paper section |
|---|---|---|
| `src/universe.py`, `src/data.py`, `src/validate.py`, `src/manifest.py` | choose the stocks; download, check and fingerprint the price data | 3 |
| `src/features.py`, `src/labels.py` | volatility scaling, encodings A and B, normalized prices, labels | 4 |
| `src/models/pattern.py` | variable-order pattern miner (`PatternMiner`) and k-nearest neighbours (`KNNModel`) | 5.1–5.2 |
| `src/models/ml.py` | LightGBM, random forest, SVM, MLP | 5.4 |
| `src/tune.py` | hyperparameter tuning with Optuna | 5.5 |
| `src/splits.py` | walk-forward folds, purging, holdout lock | 6 |
| `src/evaluate.py`, `src/models/baselines.py` | metrics, baselines, trading backtest | 7 |
| `src/losses.py`, `src/significance.py`, `src/bootstrap_ci.py` | daily losses, DM / MCS / SPA, bootstrap intervals | 8.1–8.4 |
| `src/exposure.py`, `src/break_even.py` | market exposure, break-even cost | 8.6 |
| `src/coverage.py`, `src/regimes.py`, `src/confirmatory.py` | confident days, volatility regimes, holdout test | 8.7–8.9 |
| `src/run.py` | runs the whole experiment | — |
| `ANALYSIS_PLAN.md`, `DECISIONS.md` | the frozen analysis plan and the dated decision log | — |
| `paper/` | the paper, one file per section | — |

Run the next cell to print any file here, or open it from the Files panel.

In [ ]:
FILE = "src/models/pattern.py"
print(open(FILE).read())

## Part B — re-run the whole experiment (needs the data snapshot)

This repeats the development run (18 walk-forward years, 21 models) and compares it with the committed results.

1. Joshua shares a zip of his `data/raw` folder (one `.parquet` file per ticker) with you on Google Drive. Use **Add shortcut to Drive** so it appears
   in your own Drive, and set `DATA_ZIP` below to its path.
2. Set `RUN_FULL = True` and run the cells below.

The data are checked against the SHA-256 fingerprints in `data/manifest.json` before anything runs, so the run is
refused unless the files are byte-for-byte the snapshot the study used. A fresh download from Yahoo would not
match: Yahoo revises adjusted prices as new dividends are paid.

Tuning results are cached in the repository, so the run does not repeat the hyperparameter search. On Colab's free
runtime (2 processor cores) expect about an hour, possibly more; keep the tab open. Results computed on a different
processor can differ in the last decimal places, so the comparison reports the largest difference rather than
requiring exact equality.

In [ ]:
RUN_FULL = False
DATA_ZIP = "/content/drive/MyDrive/candlestick-vs-ml-data/raw.zip"

In [ ]:
if RUN_FULL:
    from google.colab import drive
    drive.mount("/content/drive")
    import glob, shutil, zipfile
    os.makedirs("data/raw", exist_ok=True)
    with zipfile.ZipFile(DATA_ZIP) as z:
        z.extractall("/content/raw_unzipped")
    files = glob.glob("/content/raw_unzipped/**/*.parquet", recursive=True)
    for f in files:
        shutil.copy(f, "data/raw/")
    print(f"Copied {len(files)} price files into data/raw")
    !{PY} src/manifest.py --check
else:
    print("Part B is off. Set RUN_FULL = True above to run it.")

The development run must not see the holdout years. The repository's `config.yaml` was left with the holdout
unlocked after the single holdout run, so this copy is switched back to the locked setting before re-running the
development period. The change stays in this Colab copy.

In [ ]:
if RUN_FULL:
    !sed -i 's/unlock_holdout: true/unlock_holdout: false/' config.yaml
    !grep -n "unlock_holdout:" config.yaml
    !{PY} src/run.py --version colab_check

In [ ]:
if RUN_FULL:
    old = pd.read_csv("results/v4/summary.csv").set_index("model")
    new = pd.read_csv("results/colab_check/summary.csv").set_index("model").loc[old.index]
    diff = (new[["mcc", "accuracy", "ls_sharpe"]] - old[["mcc", "accuracy", "ls_sharpe"]]).abs()
    print(f"Largest difference from the committed results: MCC {diff['mcc'].max():.2e}, "
          f"accuracy {diff['accuracy'].max():.2e}, Sharpe {diff['ls_sharpe'].max():.2e}")
    display(pd.DataFrame({"MCC (committed)": old["mcc"], "MCC (this run)": new["mcc"]}).sort_values("MCC (committed)", ascending=False))